# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_json = dataset.metadata.to_json()
print(f"{getattr(dataset.metadata, 'name', 'Unnamed Dataset')}: {getattr(dataset.metadata, 'description', 'No description')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Get all record sets and their @id
record_sets = [rs['@id'] for rs in (dataset.metadata.to_json().get('recordSet', []) or [])]
print('Available Record Sets (@id):')
for rs in record_sets:
    print(f'  - {rs}')

# Show fields (columns) for each record set
if record_sets:
    for record_set_id in record_sets:
        print(f'\nFields for Record Set {record_set_id}:')
        # Access fields/columns
        record_set_obj = None
        # Find the actual record set definition
        for obj in (dataset.metadata.to_json().get('recordSet', []) or []):
            if obj['@id'] == record_set_id:
                record_set_obj = obj
                break
        fields = []
        # Fields are pointed to by 'field' or 'column' (depending on JSON-LD)
        if record_set_obj:
            fields = record_set_obj.get('field', []) or record_set_obj.get('column', []) or []
        # fields might be dict or list
        if isinstance(fields, dict):
            fields = [fields]
        for fld in fields:
            if isinstance(fld, dict):
                # Might be detailed
                print('    -', fld.get('@id', fld))
            else:
                print('    -', fld)
else:
    print('No explicit record sets found in metadata; attempting to derive from records function.')
    # Optionally, try to get the record sets via dataset.records() helper
    # This requires user inspection in the next cell.

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# If no record sets found, use '' as record_set, which loads the primary/first recordset.

# Define record set(s) to load
if record_sets:
    record_set_selection = record_sets
else:
    # Fallback in case record sets are missing; mlcroissant will load the main records
    record_set_selection = ['']

dataframes = {}
for record_set_id in record_set_selection:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} rows from record set '{record_set_id}'. Columns:")
        print(df.columns.tolist())
    except Exception as e:
        print(f"Failed to load records for record_set_id={record_set_id}: {e}")

# For demonstration, pick the first record set for analysis
if record_set_selection:
    main_record_set = record_set_selection[0]
else:
    main_record_set = ''
df0 = dataframes[main_record_set] if main_record_set in dataframes else None
if df0 is not None:
    print(df0.head())
else:
    print('No dataframe could be loaded.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Pick a numeric field for filtering/normalization
if df0 is not None:
    # Heuristic: use first numeric column if present
    numeric_cols = df0.select_dtypes(include='number').columns.tolist()
    print("Numeric columns available:", numeric_cols)
    if numeric_cols:
        numeric_field = numeric_cols[0]
        print(f"Using numeric field: {numeric_field}")
        threshold = df0[numeric_field].mean() if df0[numeric_field].dtype.kind in 'fi' else 10
        filtered_df = df0[df0[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a suitable column (heuristic: pick first object column with few unique values)
        obj_cols = df0.select_dtypes(include='object').columns.tolist()
        group_field = None
        for col in obj_cols:
            if df0[col].nunique() < 10:
                group_field = col
                break
        if group_field:
            print(f"Grouping by: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(grouped_df.head())
        else:
            print('No suitable group_field found for grouping EDA.')
    else:
        print('No numeric fields available for EDA.')
else:
    print('No DataFrame loaded for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df0 is not None and 'numeric_field' in locals():
    plt.figure(figsize=(8, 5))
    sns.histplot(df0[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
else:
    print('No numeric field available for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.